In [1]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import layers, models, callbacks

Train = '/kaggle/input/datasets/tombackert/brain-tumor-mri-data/brain-tumor-mri-dataset'

model_train = image_dataset_from_directory(
    Train,
    batch_size = 32,
    image_size = (224, 224),
    shuffle = True,
    subset = 'training',
    validation_split = 0.2,
    label_mode = 'categorical',
    seed = 123
)

model_val = image_dataset_from_directory(
    Train,
    batch_size = 32,
    image_size = (224, 224),
    shuffle = False,
    subset = 'validation',
    validation_split = 0.2,
    label_mode = 'categorical',
    seed = 123  
)

CLASS_NAMES = model_train.class_names
print("Classes: ", CLASS_NAMES)



augmentation = models.Sequential([
    layers.RandomZoom(0.2),
    layers.RandomRotation(0.1),
    
])

strategy = tf.distribute.MirroredStrategy()



with strategy.scope():
    
    main_model = models.Sequential([
        layers.Input(shape = (224, 224, 3)),
        augmentation,
        layers.Rescaling(1./255),
    
        
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
    
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
    
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
    
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(len(CLASS_NAMES), activation='softmax')
    ])
        
    
    
    main_model.compile(
        optimizer = 'adam',
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

main_model.class_names = CLASS_NAMES

reduce_speed = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-6
)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

main_model.fit(
    model_train,
    validation_data = model_val,
    epochs = 20,
    callbacks = [reduce_speed, early_stop]
)

main_model.save('brain_tumor_model.h5')



Found 7153 files belonging to 4 classes.
Using 5723 files for training.
Found 7153 files belonging to 4 classes.
Using 1430 files for validation.
Classes:  ['glioma', 'meningioma', 'notumor', 'pituitary']
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast 

## 🧠 Model Training & Architecture Summary

* **Model Type:** Custom VGG-Style Convolutional Neural Network (CNN)
* **Hardware Acceleration:** Multi-GPU Training via TensorFlow `MirroredStrategy`
* **Data Pipeline:** `image_dataset_from_directory` with 80/20 Train-Validation split & On-the-fly Data Augmentation (Zoom/Rotation)
* **Callbacks:** Dynamic Learning Rate Reduction (`ReduceLROnPlateau`) & Overfitting Prevention (`EarlyStopping`)
* **Persistence:** Saved locally as `brain_tumor_model.h5`

---
### 🔬 Next Step: Model Testing & Inference
*In the next cell, we load the trained model and perform test predictions on new MRI scans.*

In [2]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import img_to_array, load_img

model = load_model('/kaggle/working/brain_tumor_model.h5')

classes = ['Glioma', 'Meningioma', 'No tumor', 'Pituitary']

image = '/kaggle/input/datasets/tombackert/brain-tumor-mri-data/brain-tumor-mri-dataset/glioma/gl-0010.jpg'
image_upload = load_img(image, target_size=(224, 224))
image_array = np.expand_dims(img_to_array(image_upload), axis=0)

model_predict = model.predict(image_array)
model_argmax = np.argmax(model_predict)
model_possibility = np.max(model_predict) * 100

print(f'Output: {classes[model_argmax]}')
print(f'Possibility: %{model_possibility:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step
Output: Glioma
Possibility: %48.79


## 🔍 Model Evaluation & Single Image Inference

* **Model Loading:** Successfully loaded the trained weights from `brain_tumor_model.h5`.
* **Preprocessing:** Loaded and resized target MRI scan to `224x224` pixels via Keras utilities, matching the training pipeline specifications.
* **Prediction Workflow:** 
  * Expanded tensor dimensions using `np.expand_dims()` to match the batch input shape required by Keras.
  * Extracted class probability distribution using `model.predict()`.
  * Identified the final diagnosis using `np.argmax()` alongside confidence scoring via `np.max()`.